In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!rm -f pandas.py
!rm -rf pandas

!pip install --upgrade --force-reinstall pandas
!pip install --upgrade langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf

  Using cached pandas-2.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (91 kB)
  Using cached numpy-2.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.0 MB)
Using cached numpy-2.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstal

In [4]:
import os
pdf_folder = "/content/drive/MyDrive/pdfy_ml"

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
print(f"Znaleziono {len(pdf_files)} plików PDF: {pdf_files}")

Znaleziono 3 plików PDF: ['PV_29_04_241.pdf', 'eg14_cats_and_people.pdf', 'The_Origin_and_Evolution_of_Cats.pdf']


Rozpoczynam przetwarzanie plików. Najpierw ładuję treść z każdego dokumentu PDF. Następnie, w celu przygotowania danych dla modelu, dzielę cały wczytany tekst na mniejsze, nachodzące na siebie fragmenty.

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loaders = [PyPDFLoader(os.path.join(pdf_folder, f)) for f in pdf_files]
documents = []
for loader in loaders:
    documents.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(documents)
print(f"Liczba uzyskanych fragmentów: {len(chunks)}")


Liczba uzyskanych fragmentów: 223


W tym kroku przekształcam fragmenty tekstu na wektory numeryczne za pomocą wybranego modelu sentence-transformers. Następnie tworzę i indeksuję te wektory w bazie danych FAISS, co umożliwi szybkie wyszukiwanie semantyczne. Na końcu zapisuję gotowy indeks lokalnie do późniejszego wykorzystania.

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True}
)

vector_db = FAISS.from_documents(chunks, embedding_model)

vector_db.save_local("faiss_index_cats")

print("Baza wektorowa z artykułami o kotach została utworzona i zapisana")



Baza wektorowa z artykułami o kotach została utworzona i zapisana


Przeprowadzam test działania mojej bazy wektorowej. Wykonuję wyszukiwanie podobieństwa dla przykładowego zapytania, aby sprawdzić, czy system poprawnie odnajduje i zwraca 3 najbardziej relewantne fragmenty tekstu.

In [8]:
query = "Why did wildcats first start living near human settlements?"

results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n Fragment {i}:\n{doc.page_content[:400]}...")



 Fragment 1:
change in farming activities meant that quantities of grain 
were stored near human settlements. Rodents fed on the 
grain, attracting wildcats which found them an easy source 
of food. Humans grew to value these cats for protecting 
their grain stores from vermin and probably encouraged...

 Fragment 2:
thousands of years to complete. During this process, wildcats gradually adapted to human lifestyle and began to
accept human feeding (Turner and Bateson, 2014). As time went on, the morphology and behavioral
characteristics of wildcats changed, and they became more docile and close to humans (Figure 5)....

 Fragment 3:
Other scientists have proposed a different hypothesis that the domestication of cats was caused by human social
behavior. Humans accumulated a lot of garbage around their settlements, attracting many rodents and insects
(Lord, 2008). Wildcats were attracted to these places, began to contact with humans, and gradually became docile....


Teraz tworzę pełny łańcuch RAG (Retrieval-Augmented Generation). Inicjalizuję model językowy flan-t5-base i łączę go z retrieverem (moją bazą wektorową) w jeden obiekt RetrievalQA. Ten łańcuch będzie najpierw wyszukiwać kontekst, a następnie generować odpowiedź na jego podstawie.

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

qa_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=qa_pipeline)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever()
)

question = "How can cats benefit elderly people?"
response = qa_chain.invoke(question)

print(f"Pytanie: {question}")
print(f"Odpowiedź: {response['result']}")


Device set to use cpu
/tmp/ipython-input-833919972.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=qa_pipeline)


Pytanie: How can cats benefit elderly people?
Odpowiedź: Cats can be an enormous source of comfort for older people, providing some structure for each day, and a sense of purpose.


W celu poprawy jakości odpowiedzi, tworzę bardziej zaawansowaną wersję łańcucha. Definiuję własny szablon polecenia (prompt), aby precyzyjnie instruować model co do jego roli i formy odpowiedzi. Dodaję również bufor pamięci, który umożliwi prowadzenie konwersacji. Na końcu testuję tak skonfigurowany system, zadając mu serię pytań.

In [10]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory

custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an expert in feline biology, evolution, and domestication history.\n"
        "Use the context below (scientific articles and research) to provide a clear, precise, "
        "and well-structured answer.\n\n"
        "Context:\n{context}\n\n"
        "Question:\n{question}\n\n"
        "Answer:"
    )
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="question"
)

qa_chain_memory = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever(),
    chain_type="stuff",
    chain_type_kwargs={
        "prompt": custom_prompt,
        "document_variable_name": "context",
        "memory": memory
    }
)

questions = [
    "When did the earliest feline species appear?",
    "What was the role of cats in ancient Egyptian society?",
    "How did the domestication of cats begin according to research?",
    "Which wildcats are considered ancestors of modern domestic cats?",
    "Why is the evolutionary history of cats important to science?"
]

for q in questions:
    result = qa_chain_memory.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")

/tmp/ipython-input-777525002.py:17: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(



Q: When did the earliest feline species appear?
A: 50 million years ago

Q: What was the role of cats in ancient Egyptian society?
A: Egyptians liked to keep cats to control mice and other pests. Over time, cats gradually became pets and companion animals for

Q: How did the domestication of cats begin according to research?
A: A change in farming activities meant that quantities of grain Abstract Cats are one of the oldest domesticated animals in the world, with their origins dating back to the Neolithic period around 11,000 years ago.

Q: Which wildcats are considered ancestors of modern domestic cats?
A: African wildcats and European wildcats

Q: Why is the evolutionary history of cats important to science?
A: The evolutionary history of cats has great significance for our understanding of the process of animal evolution
